In [14]:
import os
import pandas as pd

import json
import sqlite3
import time
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv('/Users/aliemre2023/Desktop/apache_project/bist100_airflow/.env')


True

In [15]:
AZURE_API_KEY     = os.getenv("AZURE_API_KEY")
AZURE_ENDPOINT    = os.getenv("AZURE_ENDPOINT")
AZURE_API_VERSION = os.getenv("AZURE_API_VERSION")
AZURE_DEPLOYMENT  = os.getenv("AZURE_DEPLOYMENT_NAME")

In [17]:
DB_PATH = os.path.join(
    "/Users/aliemre2023/Desktop/apache_project/bist100_airflow/src/db/bist.db"
)

In [18]:
SECTORS = [
    "Bankacılık",
    "Sigortacılık",
    "Finansal Kiralama & Faktoring",
    "Holding & Yatırım",
    "Gayrimenkul Yatırım Ortaklığı (GYO)",
    "Otomotiv",
    "Demir Çelik & Metal",
    "Petrokimya & Rafineri",
    "Çimento & Yapı Malzemeleri",
    "Cam",
    "Gıda & İçecek",
    "Perakende & Ticaret",
    "Tekstil & Hazır Giyim",
    "Havacılık & Ulaştırma",
    "Telekomünikasyon",
    "Teknoloji & Yazılım",
    "Enerji & Elektrik",
    "Madencilik",
    "İlaç & Sağlık",
    "Dayanıklı Tüketim & Beyaz Eşya",
    "Kağıt & Ambalaj",
    "Savunma & Havacılık Sanayi",
    "Tarım & Hayvancılık",
    "Turizm & Otelcilik",
    "Medya & Eğlence",
    "Kimya",
    "Diğer Sanayi",
    "Diğer Hizmet",
]

SECTOR_ID_MAP = {s: i for i, s in enumerate(SECTORS)}

In [19]:
def get_client() -> AzureOpenAI:
    return AzureOpenAI(
        api_key=AZURE_API_KEY,
        api_version=AZURE_API_VERSION,
        azure_endpoint=AZURE_ENDPOINT,
    )


In [21]:
def _fuzzy_sector(raw_sector: str) -> str:
    raw_lower = raw_sector.lower()
    for s in SECTORS:
        if s.lower() in raw_lower or raw_lower in s.lower():
            return s
    keywords = {
        "banka": "Bankacılık", "sigorta": "Sigortacılık",
        "holding": "Holding & Yatırım", "gyo": "Gayrimenkul Yatırım Ortaklığı (GYO)",
        "otomotiv": "Otomotiv", "demir": "Demir Çelik & Metal",
        "çelik": "Demir Çelik & Metal", "petro": "Petrokimya & Rafineri",
        "rafineri": "Petrokimya & Rafineri", "çimento": "Çimento & Yapı Malzemeleri",
        "cam": "Cam", "gıda": "Gıda & İçecek", "perakende": "Perakende & Ticaret",
        "tekstil": "Tekstil & Hazır Giyim", "havacılık": "Havacılık & Ulaştırma",
        "telekom": "Telekomünikasyon", "teknoloji": "Teknoloji & Yazılım",
        "enerji": "Enerji & Elektrik", "maden": "Madencilik",
        "ilaç": "İlaç & Sağlık", "beyaz eşya": "Dayanıklı Tüketim & Beyaz Eşya",
        "kağıt": "Kağıt & Ambalaj", "savunma": "Savunma & Havacılık Sanayi",
        "tarım": "Tarım & Hayvancılık", "turizm": "Turizm & Otelcilik",
        "kimya": "Kimya",
    }
    for kw, sect in keywords.items():
        if kw in raw_lower:
            return sect
    return "Diğer Sanayi"


In [20]:
def classify_companies_batch(
    companies: list[dict],
    client: AzureOpenAI,
    batch_size: int = 15,
) -> list[dict]:

    system_prompt = f"""Sen bir Türk finans uzmanısın.
Sana BIST100 şirketlerinin kod ve ünvanlarını vereceğim.

Her şirket için aşağıdaki listeden EN UYGUN sektörü seç.
Sektör adını TAM AYNI ŞEKİLDE yaz:
{json.dumps(SECTORS, ensure_ascii=False, indent=2)}

Sadece JSON array döndür, başka bir şey yazma.
Örnek:
[
  {{"code": "MGROS", "sector": "Perakende & Ticaret"}},
  {{"code": "EREGL", "sector": "Demir Çelik & Metal"}}
]"""

    results = []

    for i in range(0, len(companies), batch_size):
        batch = companies[i:i + batch_size]
        user_msg = json.dumps(
            [{"code": c["code"], "name": c["name"]} for c in batch],
            ensure_ascii=False,
        )

        print(f"  🔄 Batch {i // batch_size + 1}/{-(-len(companies) // batch_size)}: "
              f"{[c['code'] for c in batch]}")

        try:
            response = client.chat.completions.create(
                model=AZURE_DEPLOYMENT,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_msg},
                ],
                temperature=0.0,
                max_tokens=2000,
            )

            raw = response.choices[0].message.content.strip()
            # ```json ... ``` bloğu varsa çıkar
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"):
                    raw = raw[4:]

            parsed = json.loads(raw)
            if isinstance(parsed, dict):
                for v in parsed.values():
                    if isinstance(v, list):
                        parsed = v
                        break

            for item in parsed:
                code   = item.get("code", "")
                sector = item.get("sector", "Diğer Sanayi")

                if sector not in SECTOR_ID_MAP:
                    sector = _fuzzy_sector(sector)

                results.append({
                    "code":      code,
                    "sector":    sector,
                    "sector_id": SECTOR_ID_MAP.get(sector, len(SECTORS) - 2),
                })

        except Exception as e:
            print(f"    ❌ Batch hata: {e}")
            for c in batch:
                results.append({
                    "code": c["code"],
                    "sector": "Diğer Sanayi",
                    "sector_id": SECTOR_ID_MAP["Diğer Sanayi"],
                })

        time.sleep(1)

    return results




In [22]:
def update_db(results: list[dict], db_path: str = DB_PATH):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    existing_cols = {row[1] for row in cursor.execute("PRAGMA table_info(sirket)").fetchall()}

    for col, col_def in [("sector", "TEXT DEFAULT 'Diğer Sanayi'"),
                          ("sector_id", "INTEGER DEFAULT 26")]:
        if col not in existing_cols:
            cursor.execute(f"ALTER TABLE sirket ADD COLUMN {col} {col_def}")
            print(f"  ✓ Sütun eklendi: {col}")

    for r in results:
        cursor.execute(
            "UPDATE sirket SET sector = ?, sector_id = ? WHERE sirket_code = ?",
            (r["sector"], r["sector_id"], r["code"]),
        )
        status = "✓" if cursor.rowcount > 0 else "⚠️  bulunamadı"
        print(f"  {status} {r['code']}: {r['sector']} (id={r['sector_id']})")

    conn.commit()
    conn.close()
    print(f"\n  💾 {len(results)} şirket güncellendi.")


In [23]:

def main():
    print("=" * 55)
    print("  BIST100 Şirket Sektör Sınıflandırması (GPT-4o)")
    print("=" * 55)

    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    rows = conn.execute("SELECT sirket_code, sirket_name FROM sirket ORDER BY sirket_code").fetchall()
    conn.close()

    companies = [{"code": r["sirket_code"], "name": r["sirket_name"]} for r in rows]
    print(f"\n  📋 {len(companies)} şirket bulundu\n")

    client  = get_client()
    results = classify_companies_batch(companies, client, batch_size=15)

    # Sonuç tablosu
    print(f"\n{'='*55}")
    print(f"{'Kod':<8} {'Sektör':<40} {'ID':>3}")
    print("-" * 53)
    for r in sorted(results, key=lambda x: x["code"]):
        print(f"{r['code']:<8} {r['sector']:<40} {r['sector_id']:>3}")

    # DB'ye yaz
    print(f"\n{'='*55}")
    update_db(results)

    # Dağılım
    from collections import Counter
    counts = Counter(r["sector"] for r in results)
    print(f"\n  📊 Sektör Dağılımı:")
    for sector, count in counts.most_common():
        print(f"    {sector:<40} {count:>3}")

    return results


if __name__ == "__main__":
    main()

  BIST100 Şirket Sektör Sınıflandırması (GPT-4o)

  📋 235 şirket bulundu

  🔄 Batch 1/16: ['ADEL', 'ADESE', 'ADGYO', 'AEFES', 'AGHOL', 'AGYO', 'AKBNK', 'AKCNS', 'AKENR', 'AKFGY', 'AKGRT', 'AKM, AKMEN', 'AKMGY', 'AKSGY', 'AKYHO']
  🔄 Batch 2/16: ['ALNUS, ANC', 'ANGEN', 'ARCLK', 'ARMGD', 'ARSAN', 'ARTMS', 'ARZUM', 'ASUZU', 'ATA, ATAYM', 'ATAGY', 'ATEKS', 'AVHOL', 'AVPGY', 'AVTUR', 'AYCES']
  🔄 Batch 3/16: ['AYDEM', 'AYEN', 'BAGFS', 'BAKAB', 'BERA', 'BEYAZ', 'BIMAS', 'BLS, BLSMD', 'BORLS', 'BULGS', 'BURVA', 'CATES', 'CCOLA', 'CGCAM', 'CLKMT']
  🔄 Batch 4/16: ['DENGE', 'DEVA', 'DFKTR', 'DGRVK', 'DMD, DSYAT', 'DMLKT', 'DNISI', 'DOHOL', 'DUNYH', 'DYBNK', 'DZY, DZYMK', 'EBEBK', 'ECILC', 'ECZIP', 'ECZYT']
  🔄 Batch 5/16: ['EFOR', 'EKER', 'EKGYO', 'EKIZ', 'EMIRV', 'ENJSA', 'ENSRI', 'EUHOL', 'FIN, QNBTR', 'FLAP', 'FMIZP', 'FRIGO', 'FRMPL', 'FZLGY', 'GARAN, TGB']
  🔄 Batch 6/16: ['GARFA', 'GARFL', 'GEDIK', 'GEDZA', 'GEREL', 'GLRYH', 'GLYHO', 'GOZDE', 'GSDHO', 'GUBRF', 'GUNDG', 'GWIND', 'HALKB, TH

In [24]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

rows = cursor.execute("""
    SELECT sirket_code, sirket_name, sector 
    FROM sirket 
    WHERE sector IS NULL OR sector = 'Diğer Sanayi'
    ORDER BY sirket_code
""").fetchall()
conn.close()

print(f"Eksik/default sektör: {len(rows)} şirket")
for r in rows:
    print(f"  {r[0]:<10} {r[1]}")

Eksik/default sektör: 12 şirket
  ADEL       ADEL KALEMCİLİK TİCARET VE SANAYİ A.Ş.
  BURVA      BURÇELİK VANA SANAYİ VE TİCARET A.Ş.
  FRMPL      FORMÜL PLASTİK VE METAL SANAYİ A.Ş.
  HALKB, THL TÜRKİYE HALK BANKASI A.Ş.
  HALKI, HLY HALK YATIRIM MENKUL DEĞERLER A.Ş.
  IMASM      İMAŞ MAKİNA SANAYİ A.Ş.
  KTEST      KAP TEST A.Ş.
  MAKTK      MAKİNA TAKIM ENDÜSTRİSİ A.Ş.
  MEKAG      MEKA GLOBAL MAKİNE İMALAT SANAYİ VE TİCARET A.Ş.
  MRBAS, MRS MARBAŞ MENKUL DEĞERLER A.Ş.
  NRBNK, NYB NUROL YATIRIM BANKASI A.Ş.
  OYA, OYYAT OYAK YATIRIM MENKUL DEĞERLER A.Ş.


In [25]:
fixes = {
    "ADEL":  ("Kağıt & Ambalaj", 20),
    "BURVA": ("Demir Çelik & Metal", 6),
    "FRMPL": ("Diğer Sanayi", 26),
    "HALKB": ("Bankacılık", 0),
    "HALKI": ("Finansal Kiralama & Faktoring", 2),
    "IMASM": ("Diğer Sanayi", 26),
    "KTEST": ("Diğer Hizmet", 27),
    "MAKTK": ("Diğer Sanayi", 26),
    "MEKAG": ("Diğer Sanayi", 26),
    "MRBAS": ("Finansal Kiralama & Faktoring", 2),
    "NRBNK": ("Bankacılık", 0),
    "OYA":   ("Holding & Yatırım", 3),
}

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
for code, (sector, sid) in fixes.items():
    cursor.execute(
        "UPDATE sirket SET sector = ?, sector_id = ? WHERE sirket_code = ?",
        (sector, sid, code),
    )
    status = "✓" if cursor.rowcount > 0 else "⚠️ bulunamadı"
    print(f"  {status} {code}: {sector} (id={sid})")
conn.commit()
conn.close()

  ✓ ADEL: Kağıt & Ambalaj (id=20)
  ✓ BURVA: Demir Çelik & Metal (id=6)
  ✓ FRMPL: Diğer Sanayi (id=26)
  ⚠️ bulunamadı HALKB: Bankacılık (id=0)
  ⚠️ bulunamadı HALKI: Finansal Kiralama & Faktoring (id=2)
  ✓ IMASM: Diğer Sanayi (id=26)
  ✓ KTEST: Diğer Hizmet (id=27)
  ✓ MAKTK: Diğer Sanayi (id=26)
  ✓ MEKAG: Diğer Sanayi (id=26)
  ⚠️ bulunamadı MRBAS: Finansal Kiralama & Faktoring (id=2)
  ⚠️ bulunamadı NRBNK: Bankacılık (id=0)
  ⚠️ bulunamadı OYA: Holding & Yatırım (id=3)
